In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ✅ ডিভাইস সেটআপ (GPU থাকলে CUDA ব্যবহার করবে)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ✅ ডিরেক্টরি পাথ
train_dir = "/content/drive/MyDrive/processed_data/train"
test_dir  = "/content/drive/MyDrive/processed_data/test"
model_save_path = "/content/drive/MyDrive/Model/vit_model.pth"

# ✅ ডেটা অগমেন্টেশন ও ট্রান্সফর্মেশন
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ✅ ডেটাসেট লোড
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset  = datasets.ImageFolder(root=test_dir,  transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader   = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)
num_classes   = len(train_dataset.classes)

# ✅ Vision Transformer (ViT-B/16) মডেল লোড
model = models.vit_b_16(pretrained=True, dropout=0.1)
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
model = model.to(device)

# ✅ লস ফাংশন ও অপটিমাইজার
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ✅ ট্রেনিং লুপ
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    avg_train_loss = train_loss / len(train_loader.dataset)

    # ✅ ভ্যালিডেশন
    model.eval()
    correct = total = 0
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    avg_val_loss = val_loss / len(test_loader.dataset)
    val_accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, "
          f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

# ✅ মডেল সংরক্ষণ
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")


Using device: cuda
Epoch 1/10, Train Loss: 0.6427, Val Loss: 0.6181, Val Accuracy: 65.25%
Epoch 2/10, Train Loss: 0.4311, Val Loss: 0.2714, Val Accuracy: 89.50%
Epoch 3/10, Train Loss: 0.2672, Val Loss: 0.1667, Val Accuracy: 93.25%
Epoch 4/10, Train Loss: 0.1449, Val Loss: 0.0962, Val Accuracy: 95.75%
Epoch 5/10, Train Loss: 0.1059, Val Loss: 0.1879, Val Accuracy: 94.75%
Epoch 6/10, Train Loss: 0.1989, Val Loss: 0.1417, Val Accuracy: 95.25%
Epoch 7/10, Train Loss: 0.0910, Val Loss: 0.0927, Val Accuracy: 96.75%
Epoch 8/10, Train Loss: 0.0981, Val Loss: 0.1070, Val Accuracy: 95.25%
Epoch 9/10, Train Loss: 0.0975, Val Loss: 0.1173, Val Accuracy: 95.50%
Epoch 10/10, Train Loss: 0.1053, Val Loss: 0.0603, Val Accuracy: 98.50%
Model saved to /content/drive/MyDrive/Model/vit_model.pth
